In [0]:
# ============================================
# EVENT HUBS -> ADLS BRONZE
# FINITE STRUCTURED STREAMING JOB
# ============================================


# --------------------------------------------
# 1. EVENT HUB DETAILS
# --------------------------------------------

KAFKA_BOOTSTRAP_SERVERS = (
    "banking-stream-vishal-standard.servicebus.windows.net:9093"
)

EVENT_HUB_NAME = "banking-transactions"

EVENT_HUB_CONNECTION_STRING ="--REPLACE CONNECTION STRING--"

# --------------------------------------------
# 2. EVENT HUB KAFKA CONFIGURATION
# --------------------------------------------

eh_kafka_options = {

    "kafka.bootstrap.servers":
        KAFKA_BOOTSTRAP_SERVERS,

    "subscribe":
        EVENT_HUB_NAME,

    "kafka.security.protocol":
        "SASL_SSL",

    "kafka.sasl.mechanism":
        "PLAIN",

    "kafka.sasl.jaas.config": (
        f'''
        kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required
        username="$ConnectionString"
        password="{EVENT_HUB_CONNECTION_STRING}";
        '''
    ),

    # Read the 10,000 messages already sent
    "startingOffsets":
        "earliest",

    "failOnDataLoss":
        "false"
}


# --------------------------------------------
# 3. READ EVENT HUB AS STREAM
# --------------------------------------------

df_stream = (

    spark.readStream

        .format("kafka")

        .options(**eh_kafka_options)

        .load()
)


# --------------------------------------------
# 4. CONVERT EVENT HUB MESSAGE
# --------------------------------------------

transaction_stream = (

    df_stream

        .selectExpr(

            "CAST(value AS STRING) AS transaction_data",

            "topic",

            "partition",

            "offset",

            "timestamp"
        )
)


# --------------------------------------------
# 5. ADLS BRONZE PATH
# --------------------------------------------

bronze_path = (

    "abfss://bronze@bankingdelakevishal.dfs.core.windows.net/"
    "transaction/"
)


# --------------------------------------------
# 6. CHECKPOINT PATH
# --------------------------------------------

checkpoint_path = (

    "abfss://bronze@bankingdelakevishal.dfs.core.windows.net/"
    "checkpoints/transaction_stream/"
)


# --------------------------------------------
# 7. WRITE STREAM TO BRONZE
# --------------------------------------------

query = (

    transaction_stream

        .writeStream

        .format("parquet")

        .outputMode("append")

        .option(
            "checkpointLocation",
            checkpoint_path
        )

        # Process available Event Hub messages
        # and stop automatically
        .trigger(availableNow=True)

        .start(bronze_path)
)


# --------------------------------------------
# 8. WAIT FOR COMPLETION
# --------------------------------------------

query.awaitTermination()


print(
    "Transaction streaming load completed successfully."
)

In [0]:
df_bronze = spark.read.parquet(
    "abfss://bronze@bankingdelakevishal.dfs.core.windows.net/transaction/"
)

print("Total records:", df_bronze.count())

print("Schema:", df_bronze.printSchema())

In [0]:
test_events = (
    spark.read
    .format("kafka")
    .options(**eh_kafka_options)
    .option("startingOffsets", "earliest")
    .option("endingOffsets", "latest")
    .load()
)

print("Retained Event Hub messages:", test_events.count())